In [1]:
# --- PHASE 1: Environment Setup ---
import os, sys, gc, random, time, torch
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader

# Root Setup
root = Path('/home/jupyter-1nt23cb058/Capstone')
os.chdir(root)
if str(root) not in sys.path: sys.path.insert(0, str(root))

# Custom ST-PIGNN Imports
from gnn.model import STPIGNN, LossBreakdown
import gnn.model as gnn_model
import shared.physics_config as phys_cfg

# Global Settings
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Training Hyperparameters
CFG = {
    'lr': 1e-5, 
    'max_epochs': 15, 
    'train_stride': 128, 
    'val_stride': 24,
    'total_nodes': 154902
}

def save_state(path, epoch, step, best_val):
    payload = {
        'state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scaler_state_dict': scaler_amp.state_dict(),
        'epoch': int(epoch), 'step': int(step), 'best_val_mse': float(best_val),
        'timestamp': time.ctime()
    }
    torch.save(payload, path)

print(f"✅ Phase 1 Complete: Ready on {device}.")

✅ Phase 1 Complete: Ready on cuda.


In [2]:
# --- PHASE 2: Configuration & Benchmarks ---
BASELINE = {
    'test_scaled_mse': 0.00094703699,
    'test_mae_unscaled': 5.080196,
    'test_rmse_unscaled': 6.913161
}
TARGET_SCALE = 342.9356 
print("✅ Phase 2 Complete: Benchmarks locked.")

✅ Phase 2 Complete: Benchmarks locked.


In [16]:
# --- PHASE 3: Spatial Partitioning & Hard-Alignment ---
from sklearn.cluster import KMeans

# 1. Load Topology & Global State
pyg = torch.load(root / 'data/processed/graph/topology_graph_pyg_inference.pt', weights_only=False)
edge_index_global = pyg.edge_index.long().cpu()
edge_attr_global = pyg.edge_attr.float().cpu()
train_mask_global = pyg.train_mask.bool().cpu()
num_nodes_global = int(pyg.num_nodes)

# 2. Coordinate Alignment Audit: Ensure OSMID -> NodeIndex -> Coordinates mapping is perfect
node_map_df = pd.read_parquet(root / 'data/processed/graph/topology_nodeid_to_index_map.parquet')
nodes_geo_df = pd.read_parquet(root / 'data/graphs/bangalore_utm_nodes.parquet')

# Merge to get coordinates in the EXACT order of node_index (0 to 154901)
coords_aligned = node_map_df.merge(nodes_geo_df[['osmid', 'x', 'y']], left_on='node_id', right_on='osmid', how='inner')
coords_aligned = coords_aligned.sort_values('node_index')
coords_np = coords_aligned[['x', 'y']].to_numpy(dtype='float32')

class SpatialPartition:
    def __init__(self, cid, node_ids, edge_index, edge_attr, train_mask):
        self.cid = cid
        self.n_id = torch.as_tensor(node_ids, dtype=torch.long) # Global Indices
        self.edge_index = edge_index.long()                    # Local Indices (0 to N-1)
        self.edge_attr = edge_attr.float()
        self.train_mask = train_mask.bool()
        self.upwind_edge_mask = torch.zeros(edge_index.shape[1], dtype=torch.bool)
        self.x = None 

# 3. Partitioning
print(f"Partitioning {num_nodes_global} nodes into 64 clusters...")
kmeans = KMeans(n_clusters=64, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(coords_np)
cluster_data = []
edge_index_np = edge_index_global.numpy()

for cid in range(64):
    # Get global indices belonging to this cluster
    n_ids = np.where(cluster_labels == cid)[0].astype(np.int64)
    if len(n_ids) == 0: continue
    
    # RELATIVIZATION SHIELD: Filter edges where both nodes are in this cluster
    mask_edges = np.isin(edge_index_np[0], n_ids) & np.isin(edge_index_np[1], n_ids)
    sub_edge_index = edge_index_np[:, mask_edges]
    
    # Map Global ID -> Local Index (0 to len(n_ids)-1)
    local_map = {int(global_idx): i for i, global_idx in enumerate(n_ids.tolist())}
    re_src = np.array([local_map[int(x)] for x in sub_edge_index[0]], dtype=np.int64)
    re_dst = np.array([local_map[int(x)] for x in sub_edge_index[1]], dtype=np.int64)
    
    cluster_data.append(SpatialPartition(
        cid, n_ids, 
        torch.tensor(np.stack([re_src, re_dst]), dtype=torch.long),
        torch.tensor(edge_attr_global[mask_edges], dtype=torch.float32),
        train_mask_global[torch.as_tensor(n_ids)]
    ))

print(f"✅ Phase 3 Complete: {len(cluster_data)} clusters hard-aligned.")

Partitioning 154902 nodes into 64 clusters...


/tmp/ipykernel_331123/2988614595.py:54: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(edge_attr_global[mask_edges], dtype=torch.float32),


✅ Phase 3 Complete: 64 clusters hard-aligned.


In [17]:
# --- PHASE 4: Feature Injection & Manifold Verification ---
master_df = pd.read_parquet('data/processed/graph/master_scaled_checkpoint.parquet')
pollutants = ['pm2_5_scaled', 'pm10_scaled', 'nitrogen_dioxide_scaled', 'sulphur_dioxide_scaled', 
              'carbon_monoxide_scaled', 'wind_speed_10m_scaled', 'wind_direction_10m_scaled', 
              'wind_gusts_10m_scaled', 'temperature_2m_scaled', 'relative_humidity_2m_scaled', 'surface_pressure_scaled']

# 1. Build Index Map (Ensure node_index is the key)
data_dict = {}
for node_idx, group in tqdm(master_df.groupby('node_index'), desc="Indexing Node Features"):
    vals = group.sort_values('time').tail(12)[pollutants].values
    if vals.shape[0] > 0:
        t_vals = torch.zeros((12, 11))
        actual_len = min(12, vals.shape[0])
        t_vals[:actual_len, :] = torch.from_numpy(vals[-actual_len:]).float()
        data_dict[int(node_idx)] = t_vals

# 2. Inject and Verify
total_energy = 0
for cluster in tqdm(cluster_data, desc="Injecting Manifold"):
    feat_matrix = torch.zeros((len(cluster.n_id), 12, 16))
    for i, g_id in enumerate(cluster.n_id.tolist()):
        if int(g_id) in data_dict: 
            feat_matrix[i, :, :11] = data_dict[int(g_id)]
    cluster.x = feat_matrix
    total_energy += feat_matrix.sum().item()

if total_energy == 0:
    raise ValueError("FATAL: Manifold injected with zero data. Check master_df['node_index'] alignment.")

print(f"✅ Phase 4 Complete: Manifold Energy = {total_energy:.2f}")
del master_df, data_dict; gc.collect()

Indexing Node Features:   0%|          | 0/17 [00:00<?, ?it/s]

Injecting Manifold:   0%|          | 0/64 [00:00<?, ?it/s]

✅ Phase 4 Complete: Manifold Energy = 1898.62


313

In [24]:
# --- PHASE 5: Real Temporal Sliding Window DataLoaders (Vectorized + Guarded) ---
from torch.utils.data import Dataset, DataLoader
import numpy as np
import torch

class LazyClusterDataset(Dataset):
    def __init__(self, clusters, t0, t1, window=12, stride=128, fail_on_empty=True):
        self.clusters = clusters
        self.window = int(window)
        self.starts = list(range(int(t0), int(t1) - self.window, int(stride)))
        self.fail_on_empty = bool(fail_on_empty)

        # Cache sorted global node ids per cluster for fast np.searchsorted mapping
        self.cluster_global = []
        for c in self.clusters:
            n_ids = np.asarray(c.n_id.cpu().numpy(), dtype=np.int64)
            # n_ids should already be sorted from np.where, keep as-is
            self.cluster_global.append(n_ids)

    def __len__(self):
        return len(self.starts) * len(self.clusters)

    def __getitem__(self, idx):
        t0 = self.starts[idx // len(self.clusters)]
        c_idx = idx % len(self.clusters)

        part = self.clusters[c_idx]
        n_ids = self.cluster_global[c_idx]
        num_nodes = n_ids.shape[0]

        x_win = np.zeros((self.window, num_nodes, 16), dtype=np.float32)
        y_win = np.zeros((num_nodes,), dtype=np.float32)

        # Build window
        for w in range(self.window):
            t = t0 + w
            s_ptr = time_breaks[t]
            e_ptr = time_breaks[t + 1]

            if e_ptr <= s_ptr:
                continue

            t_nodes = raw_node_indices[s_ptr:e_ptr]            # [K]
            t_vals = raw_data_values[s_ptr:e_ptr]              # [K, 17] => 16 feats + 1 target

            # Vectorized membership check
            member = np.isin(t_nodes, n_ids, assume_unique=False)
            if not np.any(member):
                continue

            g_sel = t_nodes[member]                            # global ids in this cluster
            v_sel = t_vals[member]                             # aligned values

            # Global -> local via searchsorted on cluster n_ids
            loc = np.searchsorted(n_ids, g_sel)
            valid = (loc >= 0) & (loc < num_nodes) & (n_ids[loc] == g_sel)
            if not np.any(valid):
                continue

            loc = loc[valid]
            vals = v_sel[valid]

            # Fill features
            x_win[w, loc, :] = vals[:, :16]

            # Fill target on final step
            if w == self.window - 1:
                y_win[loc] = vals[:, -1]

        # Hard guard against null windows
        if self.fail_on_empty and np.count_nonzero(x_win) == 0:
            raise RuntimeError(
                f"Empty window detected: sample_idx={idx}, cluster={c_idx}, t0={t0}. "
                "Mapping failed or source buffers are empty."
            )

        return torch.from_numpy(x_win), torch.from_numpy(y_win), c_idx

if "raw_data_values" not in globals() or "raw_node_indices" not in globals() or "time_breaks" not in globals():
    raise NameError("CRITICAL: raw_data_values/raw_node_indices/time_breaks missing. Run Cell 8 first.")

train_ds = LazyClusterDataset(cluster_data, t0=0, t1=87178, window=12, stride=128, fail_on_empty=True)
val_ds = LazyClusterDataset(cluster_data, t0=87178, t1=87514, window=12, stride=24, fail_on_empty=True)

train_loader = DataLoader(train_ds, batch_size=1, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=0)

print(f"✅ Phase 5 Complete: Real Temporal Loader active with {len(train_ds)} train samples.")

✅ Phase 5 Complete: Real Temporal Loader active with 43584 train samples.


In [19]:
# --- PHASE 6: Model Setup ---
# edge_dim is pulled from our hard-aligned clusters
model = STPIGNN(
    node_in_dim=16, 
    edge_dim=cluster_data[0].edge_attr.shape[-1], 
    spatial_hidden_dim=96, 
    temporal_hidden_dim=96, 
    gnn_layers=2
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-2)
scaler_amp = torch.amp.GradScaler('cuda')
print("✅ Phase 6 Complete: Model ready.")

✅ Phase 6 Complete: Model ready.


In [20]:
# --- PHASE 6.5: Cluster Integrity Guard ---
import torch

print("Running cluster integrity checks...")
bad = []
for part in cluster_data:
    n = int(part.x.shape[0]) if part.x is not None else 0
    e = part.edge_index
    ea = part.edge_attr
    tm = part.train_mask
    um = part.upwind_edge_mask

    if n == 0:
        bad.append((part.cid, "empty_x"))
        continue
    if e.dim() != 2 or e.shape[0] != 2:
        bad.append((part.cid, "bad_edge_index_shape"))
        continue
    if e.numel() > 0:
        e_min = int(e.min().item())
        e_max = int(e.max().item())
        if e_min < 0 or e_max >= n:
            bad.append((part.cid, f"edge_oob min={e_min} max={e_max} n={n}"))
    if ea.shape[0] != e.shape[1]:
        bad.append((part.cid, f"edge_attr_mismatch edge_attr={ea.shape[0]} edges={e.shape[1]}"))
    if tm.numel() != n:
        bad.append((part.cid, f"train_mask_mismatch mask={tm.numel()} n={n}"))
    if um.numel() != e.shape[1]:
        bad.append((part.cid, f"upwind_mask_mismatch upwind={um.numel()} edges={e.shape[1]}"))

if bad:
    print(f"CRITICAL: Found {len(bad)} malformed clusters. Training will crash if these are used.")
    for row in bad[:20]:
        print(row)
else:
    print("✅ All clusters passed integrity checks. Proceeding to Phase 7 is safe.")

Running cluster integrity checks...
✅ All clusters passed integrity checks. Proceeding to Phase 7 is safe.


In [21]:
# --- RE-INITIALIZE GLOBAL BUFFERS ---
# Run this so the DataLoader has data to pull from!

import numpy as np
import pandas as pd

# Load the raw model input
df_raw = pd.read_parquet(root / 'data/processed/model_input/model_input_node_hourly_features.parquet')
node_map = pd.read_parquet(root / 'data/processed/graph/topology_nodeid_to_index_map.parquet')
node_to_idx = dict(zip(node_map['node_id'].values, node_map['node_index'].values))
df_raw['node_index'] = df_raw['node_id'].map(node_to_idx)
df_raw = df_raw.dropna(subset=['node_index', 'timestamp']).copy()

# Scaling
df_raw['target_scaled'] = df_raw['station_pm25'].astype('float32') / 342.9356

feature_cols = [
    'station_pm10', 'station_pm25', 'station_no2', 'station_so2', 'station_co',
    'weather_wind_speed_10m', 'weather_wind_direction_10m', 'weather_wind_gusts_10m',
    'weather_temperature_2m', 'weather_relative_humidity_2m', 'weather_surface_pressure',
    'city_nitrogen_dioxide', 'city_sulphur_dioxide', 'city_pm2_5', 'city_pm10', 'city_carbon_monoxide'
]

# Extract Time Codes and Sort
all_times = pd.Index(sorted(df_raw['timestamp'].unique()))
T_total = len(all_times)
time_codes = pd.Categorical(df_raw['timestamp'], categories=all_times, ordered=True).codes.astype(np.int32)

# Build the specific buffers the NameError is looking for
raw_ts_indices = np.ascontiguousarray(time_codes)
raw_node_indices = np.ascontiguousarray(df_raw['node_index'].values)
raw_data_values = np.ascontiguousarray(df_raw[feature_cols + ['target_scaled']].to_numpy(dtype=np.float32))

sort_order = np.lexsort((raw_node_indices, raw_ts_indices))
raw_data_values = raw_data_values[sort_order]
raw_ts_indices = raw_ts_indices[sort_order]
raw_node_indices = raw_node_indices[sort_order]

# THIS IS THE MISSING VARIABLE:
time_breaks = np.searchsorted(raw_ts_indices, np.arange(T_total + 1, dtype=np.int32), side='left')

del df_raw
import gc
gc.collect()
print(f"✅ Global buffers restored. 'time_breaks' is now defined. Buffer size: {raw_data_values.shape}")

✅ Global buffers restored. 'time_breaks' is now defined. Buffer size: (1932700, 17)


In [23]:
# --- PHASE 7: Integrated Full-Manifold Training Engine (Dimension-Safe) ---
import os
import time
import gc
import numpy as np
import torch
from tqdm.auto import tqdm
from gnn.model import LossBreakdown

# 1. Configuration & Persistence
MAX_EPOCHS = 15
SAVE_INTERVAL_SECONDS = 1800
TOTAL_CLUSTERS = len(train_loader)
TOTAL_CITY_NODES = 154902
checkpoint_path = 'citywide_stpignn_checkpoint_STABLE.pt'
autosave_path = 'citywide_stpignn_autosave.pt'
best_path = 'citywide_stpignn_best.pt'

amp_enabled = (device.type == 'cuda')
scaler_amp = torch.amp.GradScaler('cuda', enabled=amp_enabled)

def save_state(path, epoch, step, best_val, loss_total=None):
    payload = {
        'state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scaler_state_dict': scaler_amp.state_dict(),
        'epoch': int(epoch),
        'step': int(step),
        'best_val_mse': float(best_val),
        'timestamp': time.ctime()
    }
    if loss_total is not None:
        payload['loss_total'] = float(loss_total)
    torch.save(payload, path)
    return payload['timestamp']

# 2. Resume Logic (Corruption-Resistant)
start_epoch = 1
start_step = 0
best_val_mse = float('inf')

potential_ckpts = [checkpoint_path, autosave_path, best_path]
resume_path = None

for path in potential_ckpts:
    if os.path.exists(path):
        try:
            torch.load(path, map_location='cpu', weights_only=True)
            resume_path = path
            break
        except Exception:
            print(f"Warning: Checkpoint {path} is corrupted. Skipping...")

if resume_path is not None:
    print(f'✅ Loading healthy checkpoint: {resume_path}')
    ckpt = torch.load(resume_path, map_location=device)
    model.load_state_dict(ckpt['state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    if 'scaler_state_dict' in ckpt:
        scaler_amp.load_state_dict(ckpt['scaler_state_dict'])
    start_epoch = int(ckpt.get('epoch', 1))
    start_step = int(ckpt.get('step', 0))
    best_val_mse = float(ckpt.get('best_val_mse', float('inf')))
    print(f'Resuming from Epoch {start_epoch} | Step {start_step}')

last_save_time = time.time()
print(f'Starting Full-Manifold Training | Scope: {TOTAL_CITY_NODES} Nodes')
print('-' * 80)

# 3. Main Loop
try:
    for epoch in range(start_epoch, MAX_EPOCHS + 1):
        model.train()
        curr_lambda = float(phys_cfg.PHYSICS_LOSS_LAMBDA) * min(1.0, epoch / 12.0)

        pbar = tqdm(total=TOTAL_CLUSTERS, desc=f'Epoch {epoch}', unit='cluster')
        
        # Audit: xb_raw and yb_raw come from Phase 5 Real Loader
        for i, (xb_raw, yb_raw, c_idx_tensor) in enumerate(train_loader):
            if epoch == start_epoch and i < start_step:
                pbar.update(1)
                continue

            # FIX: Handle DataLoader list output for c_idx
            c_idx = int(c_idx_tensor[0].item())
            part = cluster_data[c_idx]

            # --- DIMENSION ALIGNMENT & SENTRY ---
            # Real Loader returns [B, T, N, F]. part.x is [N, T, F]
            xb = xb_raw.to(device, non_blocking=True)
            yb = yb_raw.to(device, non_blocking=True)
            num_nodes = xb.shape[2]

            # CPU Sentry: Validate bounds before GPU operation
            if part.edge_index.numel() > 0:
                if part.edge_index.max() >= num_nodes:
                    pbar.update(1)
                    continue 

            edge_i = part.edge_index.to(device)
            edge_a = part.edge_attr.to(device)
            
            # Identify Tiered Mode (Audit: Channels 0-4 carry ground truth signal)
            data_signal = (xb[0, :, :, :5].sum() > 0)
            mask = part.train_mask.to(device=device, dtype=torch.bool)
            u_mask = part.upwind_edge_mask.to(device=device, dtype=torch.bool)

            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast(device_type='cuda', enabled=amp_enabled):
                pred = model(x_seq=xb, edge_index=edge_i, edge_attr=edge_a)
                target = yb # Already shaped correctly by Real Loader

                if data_signal and bool(mask.any()):
                    # TIER 1: STATION MODE (Supervised Data + Physics)
                    res = gnn_model.compute_total_loss(
                        pred=pred, target=target, train_mask=mask,
                        edge_index=edge_i, edge_attr=edge_a,
                        upwind_edge_mask=u_mask, physics_lambda=curr_lambda,
                    )
                    mode_label = 'STATION'
                else:
                    # TIER 2: PHYS-ONLY MODE (Road network constraints)
                    phys_p = gnn_model.physics_upwind_penalty(
                        pred=pred, edge_index=edge_i,
                        upwind_edge_mask=u_mask, edge_attr=edge_a,
                    )
                    res = LossBreakdown(total=curr_lambda * phys_p, data=pred.new_tensor(0.0), physics=phys_p)
                    mode_label = 'PHYS'

            # --- BACKPROP ---
            if torch.isfinite(res.total) and res.total > 0:
                scaler_amp.scale(res.total).backward()
                scaler_amp.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
                scaler_amp.step(optimizer)
                scaler_amp.update()

            # --- UI REPORTING ---
            t_l = float(res.total.detach().cpu().item())
            d_l = float(res.data.detach().cpu().item())
            p_l = float(res.physics.detach().cpu().item())
            
            pbar.set_postfix({
                'M': mode_label,
                'T': f'{t_l:.4e}',
                'D': f'{d_l:.4e}',
                'P': f'{p_l:.4e}',
                'Step%': f'{(i / TOTAL_CLUSTERS) * 100:.1f}%'
            })
            pbar.update(1)

            # Periodic VRAM Flush
            if i % 100 == 0:
                del xb, yb, pred, res
                torch.cuda.empty_cache()
                gc.collect()

            # Background Autosave
            if (time.time() - last_save_time) > SAVE_INTERVAL_SECONDS:
                save_state(autosave_path, epoch, i, best_val_mse)
                last_save_time = time.time()

        pbar.close()
        save_state(checkpoint_path, epoch + 1, 0, best_val_mse)

        # --- VALIDATION PHASE ---
        model.eval()
        val_losses = []
        with torch.no_grad():
            for vxb_raw, vyb_raw, cv_idx_t in val_loader:
                v_idx = int(cv_idx_t[0].item())
                v_part = cluster_data[v_idx]
                vxb = vxb_raw.to(device)
                v_mask = (v_part.val_mask if hasattr(v_part, 'val_mask') else v_part.train_mask).to(device, dtype=torch.bool)
                if not v_mask.any(): continue
                
                with torch.amp.autocast(device_type='cuda', enabled=amp_enabled):
                    vp = model(x_seq=vxb, edge_index=v_part.edge_index.to(device), edge_attr=v_part.edge_attr.to(device))
                    v_res = gnn_model.compute_total_loss(
                        pred=vp, target=vyb_raw.to(device), train_mask=v_mask,
                        edge_index=v_part.edge_index.to(device),
                        edge_attr=v_part.edge_attr.to(device),
                        upwind_edge_mask=v_part.upwind_edge_mask.to(device),
                        physics_lambda=curr_lambda
                    )
                    if torch.isfinite(v_res.total):
                        val_losses.append(v_res.total.item())
        
        avg_val = np.mean(val_losses) if val_losses else 0.0
        print(f'\n>>> EPOCH {epoch} | Safe Val Total: {avg_val:.6e}')
        
        if 0 < avg_val < best_val_mse:
            best_val_mse = avg_val
            save_state(best_path, epoch, 0, best_val_mse)
            print(f'[SAVE] New best validation: {best_val_mse:.6e}')

        start_step = 0

except KeyboardInterrupt:
    print('\nKeyboard Interrupt: Emergency Save...')
    save_state(checkpoint_path, epoch, i if 'i' in locals() else 0, best_val_mse)

print('Training session complete.')

✅ Loading healthy checkpoint: citywide_stpignn_checkpoint_STABLE.pt
Resuming from Epoch 2 | Step 5352
Starting Full-Manifold Training | Scope: 154902 Nodes
--------------------------------------------------------------------------------


Epoch 2:   0%|          | 0/43584 [00:00<?, ?cluster/s]


Keyboard Interrupt: Emergency Save...
Training session complete.


In [25]:
# --- PHASE 8.5: Data Pipeline Audit Cell (Run before Cell 9) ---
import numpy as np
import torch

print("=== BUFFER AUDIT ===")
print("raw_data_values shape:", raw_data_values.shape)
print("raw_node_indices shape:", raw_node_indices.shape)
print("time_breaks len:", len(time_breaks))
print("nonzero raw_data_values:", int(np.count_nonzero(raw_data_values)))
print("finite ratio raw_data_values:", float(np.isfinite(raw_data_values).mean()))

print("\n=== CLUSTER AUDIT ===")
sensor_clusters = 0
upwind_clusters = 0
for c in cluster_data:
    if int(c.train_mask.sum().item()) > 0:
        sensor_clusters += 1
    if int(c.upwind_edge_mask.sum().item()) > 0:
        upwind_clusters += 1
print("clusters with sensors:", sensor_clusters, "/", len(cluster_data))
print("clusters with upwind edges:", upwind_clusters, "/", len(cluster_data))

print("\n=== SAMPLE WINDOW AUDIT ===")
def inspect_sample(ds, idx):
    x, y, cidx = ds[idx]
    nz_x = int(torch.count_nonzero(x).item())
    nz_y = int(torch.count_nonzero(y).item())
    cid = int(cidx)
    part = cluster_data[cid]
    print({
        "sample_idx": idx,
        "cluster": cid,
        "x_shape": tuple(x.shape),
        "y_shape": tuple(y.shape),
        "x_nonzero": nz_x,
        "y_nonzero": nz_y,
        "train_mask_true": int(part.train_mask.sum().item()),
        "upwind_true": int(part.upwind_edge_mask.sum().item()),
    })

# probe a few samples across epoch space
probe_idxs = [0, len(train_ds)//4, len(train_ds)//2, (3*len(train_ds))//4, len(train_ds)-1]
for pi in probe_idxs:
    inspect_sample(train_ds, int(pi))

print("\n✅ Audit complete. If x_nonzero is still 0, mapping is broken. If upwind_true is 0 everywhere, physics loss will be 0 by design.")

=== BUFFER AUDIT ===
raw_data_values shape: (1932700, 17)
raw_node_indices shape: (1932700,)
time_breaks len: 87851
nonzero raw_data_values: 32771114
finite ratio raw_data_values: 0.8101604278074866

=== CLUSTER AUDIT ===
clusters with sensors: 16 / 64
clusters with upwind edges: 0 / 64

=== SAMPLE WINDOW AUDIT ===


RuntimeError: Empty window detected: sample_idx=0, cluster=0, t0=0. Mapping failed or source buffers are empty.